In [ ]:
!pip install lightgbm

In [ ]:
import numpy as np, pandas as pd, gc, joblib, inspect
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score

In [ ]:
BASE       = '/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
MODELS_DIR = f'{BASE}/amia/models'
MATRIX_DIR = f'{BASE}/amia/matrix'
TF         = 24                 
VER_BASE   = 'ehrdemo'
VER_SURV   = 'ehrdemo_survnobasic'
LABEL_COL  = 'IsPositive'       
PREFIX     = {'lightgbm':'LGB', 'randomforest':'RF', 'logreg':'LR', 'mlp':'MLP', 'xgboost':'XGB'}

In [ ]:
def load_test(ver, tf):
    rs = pd.read_parquet(f'{BASE}/amia/risk_scores/LGB_{ver}_{tf}m.parquet')  
    test_ids = set(rs.loc[rs['split'] == 'test', 'person_id'])
    df = pd.read_parquet(f'{MATRIX_DIR}/matrix_{ver}_{tf}.parquet')
    df = df[df['person_id'].isin(test_ids)].reset_index(drop=True)
    y  = df[LABEL_COL].values
    X  = df.drop(columns=['person_id', LABEL_COL])
    del df; gc.collect()
    return X, y

def load_parts(model_type, ver, tf):
    art = joblib.load(f'{MODELS_DIR}/{PREFIX[model_type]}_{ver}_{tf}m.joblib')  
    if not isinstance(art, dict):
        return None, None, art, None
    model  = art.get('model') or art.get('clf') or art.get('estimator')
    imp    = art.get('imputer') or art.get('imp')
    scaler = art.get('scaler')
    cols   = art.get('feature_cols') or art.get('features') or art.get('columns')
    return imp, scaler, model, cols

def make_proba(imp, scaler, model):
    mod = type(model).__module__
    if hasattr(model, 'predict_proba'):                 
        core = lambda A: model.predict_proba(A)[:, 1]
    elif mod.startswith('lightgbm'):                    
        core = lambda A: np.asarray(model.predict(A))
    elif mod.startswith('xgboost'):                     
        import xgboost as xgb
        core = lambda A: np.asarray(model.predict(xgb.DMatrix(A)))
    else:
        raise TypeError(f'no proba for {type(model)}')
    def fn(X):
        A = X
        if imp    is not None: A = imp.transform(A)
        if scaler is not None: A = scaler.transform(A)
        return core(A)
    return fn

def build_groups(cols):
    groups = {}
    for c in cols:
        if c.startswith('surv_'):
            if c in ('surv_total_count','surv_unique_question_count','surv_skip_count'):
                groups.setdefault('survey_response_volume', []).append(c)
            elif '__' in c:
                groups.setdefault(c.split('__')[0], []).append(c)
            else:
                groups.setdefault(c, []).append(c)
        elif c.startswith(('lab_','meas_')) and (c.endswith('_mean') or c.endswith('_last')):
            groups.setdefault(c[:c.rfind('_')], []).append(c)
        else:
            groups.setdefault(c, []).append(c)
    return groups

from tqdm.auto import tqdm

def block_perm(proba_fn, X_test, y_test, groups, n_repeats=5, seed=42):
    base = average_precision_score(y_test, proba_fn(X_test))
    rng  = np.random.default_rng(seed)
    Xw   = X_test.copy()
    rows = []
    for name, cols in tqdm(list(groups.items()), desc='blocks'):   
        cols = [c for c in cols if c in Xw.columns]
        if not cols: continue
        block, drops = Xw[cols].to_numpy().copy(), []
        for _ in range(n_repeats):
            perm = rng.permutation(len(Xw))
            Xw[cols] = block[perm]
            drops.append(base - average_precision_score(y_test, proba_fn(Xw)))
        Xw[cols] = block
        rows.append({'block':name,'n_cols':len(cols),
                     'drop_mean':float(np.mean(drops)),'drop_std':float(np.std(drops))})
    return pd.DataFrame(rows).sort_values('drop_mean', ascending=False).reset_index(drop=True)

In [ ]:
imp, scaler, model, cols = load_parts('lightgbm', VER_SURV, TF)
Xte, yte = load_test(VER_SURV, TF)
Xte = Xte[cols].astype('float32') if cols is not None else Xte.astype('float32')
proba = make_proba(imp, scaler, model)

base_check = average_precision_score(yte, proba(Xte))
print('base PR-AUC (LGB', VER_SURV, TF, 'm) =', round(float(base_check), 4))

In [ ]:
def run_tabular(model_type, tf, n_repeats=5):
    out = {}
    for tag, ver in (('ehr', VER_BASE), ('sp', VER_SURV)):
        imp, scaler, model, cols = load_parts(model_type, ver, tf)
        if cols is None and type(model).__module__.startswith('lightgbm'):
            feat = model.feature_name()
            if feat and not feat[0].startswith('Column_'):
                cols = feat
        Xte, yte = load_test(ver, tf)
        Xte = Xte[cols].astype('float32') if cols is not None else Xte.astype('float32')
        proba = make_proba(imp, scaler, model)
        base  = average_precision_score(yte, proba(Xte))
        top   = block_perm(proba, Xte, yte, build_groups(Xte.columns), n_repeats=n_repeats).head(10)
        out[tag] = {'base_pr_auc': float(base), 'top10': top}
        del model, Xte; gc.collect()
    sp = out['sp']['top10'].copy()
    sp['is_survey'] = sp['block'].str.startswith('surv_') | sp['block'].eq('survey_response_volume')
    print(f"\n===== {model_type} (tf={tf}m) =====")
    print(f"base PR-AUC  ehr={out['ehr']['base_pr_auc']:.4f}  +surv={out['sp']['base_pr_auc']:.4f}")
    print("\n[baseline EHR top-10]")
    print(out['ehr']['top10'][['block','drop_mean','drop_std']].to_string(index=False))
    print("\n[EHR+Survey top-10]")
    print(sp[['block','drop_mean','drop_std','is_survey']].to_string(index=False))
    print(f"\nsurvey 占 +survey top-10: {sp['is_survey'].mean():.0%}  "
          f"survey 块: {sp.loc[sp['is_survey'],'block'].tolist()}")
    out['sp']['top10'] = sp
    return out

assert 'make_proba' in inspect.getsource(run_tabular), 
print('Cell 3 OK')

In [ ]:
res_lgbm = run_tabular('lightgbm', TF)

In [ ]:
sp = res_lgbm['sp']['top10']
content = sp[~sp['block'].eq('survey_response_volume')].head(10).copy()   
content['is_survey_content'] = content['block'].str.startswith('surv_')
print(content[['block','drop_mean','drop_std','is_survey_content']].to_string(index=False))

In [ ]:
def build_groups_mixed(cols):
    groups = {}
    for c in cols:
        if c.startswith('surv_'):
            if c in ('surv_total_count','surv_unique_question_count','surv_skip_count'):
                groups.setdefault('survey_response_volume', []).append(c)
            elif '__' in c:
                groups.setdefault(c.split('__')[0], []).append(c)
            else:
                groups.setdefault(c, []).append(c)
        elif c.startswith('lab_'):
            groups.setdefault('lab_seq', []).append(c)
        elif c.startswith('drug_'):
            groups.setdefault('drug_seq', []).append(c)
        elif c.startswith('cond_'):
            groups.setdefault('condition_seq', []).append(c)
        elif c.startswith('meas_'):
            groups.setdefault('measurement_seq', []).append(c)
        elif c.startswith('obs_'):
            groups.setdefault('observation_seq', []).append(c)
        elif c.startswith('demo_'):
            groups.setdefault('demographic', []).append(c)
        else:
            groups.setdefault(c, []).append(c)
    return groups

groups_mixed = build_groups_mixed(Xte.columns)
covered = sum(groups_mixed.values(), [])
assert len(covered) == len(set(covered)) == len(Xte.columns), 
print('group number:', len(groups_mixed), 'survey-related:',
      sum(1 for k in groups_mixed if k.startswith('surv') or k == 'survey_response_volume'))

mixed = block_perm(proba, Xte, yte, groups_mixed, n_repeats=5)
mixed['is_survey'] = mixed['block'].str.startswith('surv_') | mixed['block'].eq('survey_response_volume')

In [ ]:
import os
os.makedirs(f'{BASE}/amia/perm_importance', exist_ok=True)
mixed.to_csv(f'{BASE}/amia/perm_importance/LGB_{VER_SURV}_{TF}m_mixed_granularity.csv', index=False)
mixed_no_vol = mixed[mixed['block'] != 'survey_response_volume'].reset_index(drop=True)
print(mixed_no_vol.to_string(index=False))
mixed_no_vol.to_csv(f'{BASE}/amia/perm_importance/LGB_{VER_SURV}_{TF}m_mixed_granularity_no_volume.csv', index=False)

In [ ]:
groups_all_agg = build_groups_mixed(Xte.columns).copy()
survey_keys = [k for k in groups_all_agg if (k.startswith('surv_') or k == 'survey_response_volume')]
survey_content_cols = sum((groups_all_agg.pop(k) for k in survey_keys if k != 'survey_response_volume'), [])
groups_all_agg.pop('survey_response_volume', None)          
groups_all_agg['survey_content'] = survey_content_cols

covered = sum(groups_all_agg.values(), [])
assert len(covered) == len(set(covered)), 'repeated'

all_agg = block_perm(proba, Xte, yte, groups_all_agg, n_repeats=5)
all_agg['is_survey'] = all_agg['block'] == 'survey_content'

print(all_agg.to_string(index=False))
all_agg.to_csv(f'{BASE}/amia/perm_importance/LGB_{VER_SURV}_{TF}m_all_categories_agg_no_volume.csv', index=False)

In [ ]:
imp, scaler, model, cols = load_parts('lightgbm', VER_SURV, TF)
Xte, yte = load_test(VER_SURV, TF)
Xte = Xte[cols].astype('float32') if cols is not None else Xte.astype('float32')
proba = make_proba(imp, scaler, model)

vol_cols     = ['surv_total_count','surv_unique_question_count','surv_skip_count']
content_cols = [c for c in Xte.columns if c.startswith('surv_') and c not in vol_cols]
two = block_perm(proba, Xte, yte,
                 {'survey_volume_only': vol_cols, 'survey_content_only': content_cols},
                 n_repeats=5)
print(two.to_string(index=False))

In [ ]:
for stub in ['surv_within_the_past_12_months_were',
             'surv_disability_difficulty_concentr',
             'surv_recreational_drug_use_which_dr']:
    hit = [c for c in Xte.columns if c.startswith(stub)]
    print(stub, '->', hit)

In [ ]:
import glob
nm_files = sorted(glob.glob(f'{BASE}/amia/feature/survey_name_map_24*'))
print(nm_files)
nm = pd.concat([pd.read_csv(f) for f in nm_files], ignore_index=True)   
for stub in ['surv_within_the_past_12_months_were',
             'surv_disability_difficulty_concentr',
             'surv_recreational_drug_use_which_dr']:
    print('\n===', stub, '===')
    print(nm[nm.iloc[:,0].astype(str).str.startswith(stub)].to_string(index=False))

In [ ]:
res_rf = run_tabular('randomforest', TF)

In [ ]:
imp, scaler, model, cols = load_parts('lightgbm', VER_SURV, TF)
Xte, yte = load_test(VER_SURV, TF)
Xte = Xte[cols].astype('float32') if cols is not None else Xte.astype('float32')
proba = make_proba(imp, scaler, model)

vol_cols     = ['surv_total_count','surv_unique_question_count','surv_skip_count']
opioid_cols  = [c for c in Xte.columns if c.startswith('surv_recreational_drug_use_which_dr')
                and (c.endswith('_pre') or c.endswith('_str'))]        
drug_nonop   = [c for c in Xte.columns if c.startswith('surv_recreational_drug_use_which_dr')
                and c not in opioid_cols]
content_all  = [c for c in Xte.columns if c.startswith('surv_') and c not in vol_cols]
content_noop = [c for c in content_all if c not in opioid_cols]        

groups = {
    'survey_volume_only'      : vol_cols,
    'survey_content_all'      : content_all,     
    'survey_content_no_opioid': content_noop,    
    'drug_opioid_items_only'  : opioid_cols,     
    'drug_nonopioid_items'    : drug_nonop,
}
print(block_perm(proba, Xte, yte, groups, n_repeats=5).to_string(index=False))

In [ ]:
def block_perm_indep(proba_fn, X_test, y_test, cols, n_repeats=5, seed=42):
    base = average_precision_score(y_test, proba_fn(X_test))
    rng, Xw, drops = np.random.default_rng(seed), X_test.copy(), []
    arr = Xw[cols].to_numpy().copy()
    for _ in range(n_repeats):
        for j in range(arr.shape[1]):
            Xw[cols[j]] = arr[rng.permutation(len(Xw)), j]
        drops.append(base - average_precision_score(y_test, proba_fn(Xw)))
        Xw[cols] = arr
    return float(np.mean(drops))

print('block:', 0.1566)  
print('indep :', block_perm_indep(proba, Xte, yte, content_noop))

In [ ]:
import pandas as pd
p = f'{BASE}/amia/feature/cond_features_6.parquet'  
df = pd.read_parquet(p)
print('shape:', df.shape)
print('top 30:', df.columns[:30].tolist())
print('last 10:', df.columns[-10:].tolist())
print('with person_id:', 'person_id' in df.columns, '| with label:', 
      [c for c in df.columns if c.lower() in ('ispositive','is_positive','y','label')])

In [ ]:
Sequentail RNN

In [ ]:
import numpy as np, pandas as pd, torch, joblib, gc, json, os
import torch.nn as nn
from sklearn.metrics import average_precision_score
from tqdm.auto import tqdm

class SeqTower(nn.Module):
    def __init__(s, kind, d_seq, d_static, K, hs=128, ht=64, p=0.3, d_model=128, nhead=4, nlayers=2):
        super().__init__(); s.kind = kind
        if kind in ('LSTM', 'GRU'):
            s.enc = (nn.LSTM if kind == 'LSTM' else nn.GRU)(d_seq, hs, batch_first=True); s.out = hs
        else:
            s.proj = nn.Linear(d_seq, d_model); s.pos = nn.Parameter(torch.randn(K, d_model) * 0.02)
            s.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(d_model, nhead, 256, p, batch_first=True), nlayers); s.out = d_model
        s.stat = nn.Sequential(nn.Linear(d_static, ht), nn.ReLU(), nn.Dropout(p))
        s.head = nn.Sequential(nn.Linear(s.out + ht, 64), nn.ReLU(), nn.Dropout(p), nn.Linear(64, 1))
    def forward(s, xs, xt, lens):
        lens = lens.to(torch.long)
        if s.kind in ('LSTM', 'GRU'):
            packed = nn.utils.rnn.pack_padded_sequence(xs, lens.cpu(), batch_first=True, enforce_sorted=False)
            if s.kind == 'LSTM': _, (h, _) = s.enc(packed)
            else:                _, h = s.enc(packed)
            hseq = h[-1]
        else:
            kpm = torch.arange(xs.size(1), device=xs.device).unsqueeze(0) >= lens.to(xs.device).unsqueeze(1)
            z = s.enc(s.proj(xs) + s.pos, src_key_padding_mask=kpm)
            w = (~kpm).float().unsqueeze(-1)
            hseq = (z * w).sum(1) / w.sum(1).clamp(min=1e-6)
        return s.head(torch.cat([hseq, s.stat(xt)], 1)).squeeze(1)

BASE  = '/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
SEQ_DIR = f'{BASE}/amia/seq_data'; MODEL_DIR = f'{BASE}/amia/models'
TF = 24; PREFIX = 'RNN'; VER = 'ehrdemo_survnobasic'; label = f'{VER}_{TF}m'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', device)

ckpt = torch.load(f'{MODEL_DIR}/{PREFIX}_{label}.pt', map_location=device)
K = ckpt['K']; D_seq = ckpt['D_seq']; d_static = ckpt['d_static']
model = SeqTower(ckpt['kind'], D_seq, d_static, K).to(device)
model.load_state_dict(ckpt['state_dict']); model.eval()
print('kind:', ckpt['kind'], '| K:', K, '| D_seq:', D_seq, '| d_static:', d_static)

static_prep = joblib.load(f'{SEQ_DIR}/static_prep_{TF}.joblib')
static_cols = static_prep['surv']['cols']
print('static_cols n =', len(static_cols))

In [ ]:
g = np.load(f'{SEQ_DIR}/grid_{TF}.npz')
persons = g['person_id']; y = g['y'].astype('float32'); split = g['split']; nr = g['n_real'].astype('int64')
N = len(persons)

R, Sl, F, V, off = [], [], [], [], 0
modality_range = {}
for mod in ['condition', 'drug', 'observation', 'lab', 'measurement']:
    z = np.load(f'{SEQ_DIR}/seq_{mod}_{TF}.npz'); D = int(z['D'])
    modality_range[mod] = (off, off + D)
    R.append(z['row']); Sl.append(z['slot']); F.append(z['feat'].astype('int32') + off); V.append(z['val'])
    off += D
row = np.concatenate(R); slot = np.concatenate(Sl).astype('int64')
feat = np.concatenate(F).astype('int64'); val = np.concatenate(V)
o = np.argsort(row, kind='stable'); row, slot, feat, val = row[o], slot[o], feat[o], val[o]
rp = np.searchsorted(row, np.arange(N + 1)).astype('int64')
del R, Sl, F, V, o; gc.collect()

Sm = np.load(f'{SEQ_DIR}/static_{TF}.npz')
static = Sm['static_surv'] if VER.endswith('survnobasic') else Sm['static_nosurv']

te = np.where(split == 2)[0]
static_test = static[te]
print('modality_range:', modality_range)
print('D_seq check:', off, '==', D_seq)
print('n_test:', len(te))

In [ ]:
def build_seq_batch(rows, rp, sl, ft, vl, K, D_seq, group_mask=None, perm_rows=None):
    B = len(rows)
    xs = np.zeros((B, K, D_seq), dtype='float32')
    for i, r in enumerate(rows):
        r = int(r); a, b = int(rp[r]), int(rp[r+1])
        ft = feat[a:b]
        fr, sr, vr = ft, slot[a:b], val[a:b]
        if group_mask is None:
            xs[i, sr, fr] = vr
        else:
            keep = ~group_mask[fr]
            xs[i, sr[keep], fr[keep]] = vr[keep]
            r2 = int(perm_rows[i]); a2, b2 = int(rp[r2]), int(rp[r2+1])
            fr2, sr2, vr2 = feat[a2:b2], slot[a2:b2], val[a2:b2]
            g2 = group_mask[fr2]
            xs[i, sr2[g2], fr2[g2]] = vr2[g2]
    return xs

def predict_proba_seq(row_ids, static_rows, nr_arr, group_mask=None, perm_row_ids=None, batch_size=2048):
    probs = np.empty(len(row_ids), dtype=np.float32)
    with torch.no_grad():
        for i in range(0, len(row_ids), batch_size):
            rb  = row_ids[i:i+batch_size]
            pb  = perm_row_ids[i:i+batch_size] if perm_row_ids is not None else None
            xs  = build_seq_batch(rb, rp, slot, feat, val, K, D_seq, group_mask, pb)
            st  = static_rows[i:i+batch_size]
            ln  = np.maximum(nr[rb], 1)                      
            logit = model(
                torch.from_numpy(xs).to(device, non_blocking=True),
                torch.from_numpy(st.astype('float32')).to(device, non_blocking=True),
                torch.from_numpy(ln.astype('int64')).to(device, non_blocking=True),
            )
            probs[i:i+batch_size] = torch.sigmoid(logit).cpu().numpy()
    return probs

static_test = static[te]                              
base = average_precision_score(y[te], predict_proba_seq(te, static_test, nr))
print('base PR-AUC (RNN', label, ') =', round(float(base), 4))

In [ ]:
def build_groups_static_mixed(cols):
    groups = {}
    for c in cols:
        if c.startswith('surv_'):
            if c in ('surv_total_count','surv_unique_question_count','surv_skip_count'):
                groups.setdefault('survey_response_volume', []).append(c)
            elif '__' in c:
                groups.setdefault(c.split('__')[0], []).append(c)
            else:
                groups.setdefault(c, []).append(c)
        elif c.startswith('demo_'):
            groups.setdefault('demo_age' if c == 'demo_age' else 'demographic', []).append(c)
        elif c.startswith('lab_'):
            groups.setdefault('lab_static', []).append(c)
        elif c.startswith('meas_'):
            groups.setdefault('measurement_static', []).append(c)
        else:
            groups.setdefault(c, []).append(c)
    return groups

static_groups_fixed = build_groups_static_mixed(static_cols)
covered = sum(static_groups_fixed.values(), [])
assert len(covered) == len(set(covered)) == len(static_cols), 'static repated'
print('static groups number:', len(static_groups_fixed))

In [ ]:
def block_perm_static_only(groups, n_repeats=5, seed=42, batch_size=2048):
    rng = np.random.default_rng(seed)
    n = len(te); rows = []
    idx_of = {c: i for i, c in enumerate(static_cols)}
    sm = static_test.copy()
    for name, cols in tqdm(list(groups.items()), desc='static blocks (fixed)'):
        idx = [idx_of[c] for c in cols if c in idx_of]
        if not idx: continue
        block, drops = sm[:, idx].copy(), []
        for _ in range(n_repeats):
            perm = rng.permutation(n)
            sm[:, idx] = block[perm]
            drops.append(base - average_precision_score(y[te], predict_proba_seq(te, sm, nr, batch_size=batch_size)))
        sm[:, idx] = block
        rows.append({'block': name, 'n_cols': len(idx), 'tower': 'static',
                     'drop_mean': float(np.mean(drops)), 'drop_std': float(np.std(drops))})
    return pd.DataFrame(rows)

static_fixed = block_perm_static_only(static_groups_fixed, n_repeats=5)

In [ ]:
import pandas as pd

recovered = pd.read_csv(f'{BASE}/amia/perm_importance/RNN_{VER}_{TF}m_mixed_granularity_no_volume_FIXED.csv')
print(recovered.columns.tolist())
print(recovered[recovered['tower'] == 'sequence'])

In [ ]:
seq_part = recovered[recovered['tower'] == 'sequence'].reset_index(drop=True)
print(seq_part)

In [ ]:
top_rnn_fixed = pd.concat([static_fixed, seq_part], ignore_index=True).sort_values('drop_mean', ascending=False).reset_index(drop=True)
top_rnn_fixed['is_survey'] = top_rnn_fixed['block'].str.startswith('surv_') | top_rnn_fixed['block'].eq('survey_response_volume')

top_rnn_no_vol_fixed = top_rnn_fixed[top_rnn_fixed['block'] != 'survey_response_volume'].reset_index(drop=True)
print(top_rnn_no_vol_fixed.to_string(index=False))

top_rnn_no_vol_fixed.to_csv(f'{BASE}/amia/perm_importance/RNN_{VER}_{TF}m_mixed_granularity_no_volume_FIXED.csv', index=False)
print('saved')

In [ ]:
demo_all_idx = [i for i, c in enumerate(static_cols) if c.startswith('demo_')]
demographic_merged = block_perm_static_group(demo_all_idx, 'demographic', n_repeats=5)
print(demographic_merged)

def replace_demo(df):
    out = df[~df['block'].isin(['demo_age','demographic'])].copy()
    out = pd.concat([out, pd.DataFrame([demographic_merged])], ignore_index=True)
    return out.sort_values('drop_mean', ascending=False).reset_index(drop=True)

top_rnn_no_vol_fixed_v2 = replace_demo(top_rnn_no_vol_fixed)
top_rnn_no_vol_fixed_v2['is_survey'] = top_rnn_no_vol_fixed_v2['block'].str.startswith('surv_')
print(top_rnn_no_vol_fixed_v2.to_string(index=False))
top_rnn_no_vol_fixed_v2.to_csv(f'{BASE}/amia/perm_importance/RNN_{VER}_{TF}m_mixed_granularity_no_volume_FIXED_v2.csv', index=False)

In [ ]:
survey_all_cols_idx = [i for i, c in enumerate(static_cols) if c.startswith('surv_') and c not in
                       ('surv_total_count','surv_unique_question_count','surv_skip_count')]

rnn_survey_agg = block_perm_static_group(survey_all_cols_idx, 'survey_content', n_repeats=5)
print(rnn_survey_agg)
rnn_all_agg = pd.concat([
    top_rnn_no_vol_fixed[~top_rnn_no_vol_fixed['block'].str.startswith('surv_')],
    pd.DataFrame([rnn_survey_agg])
], ignore_index=True).sort_values('drop_mean', ascending=False).reset_index(drop=True)
rnn_all_agg['is_survey'] = rnn_all_agg['block'] == 'survey_content'

print(rnn_all_agg.to_string(index=False))
rnn_all_agg.to_csv(f'{BASE}/amia/perm_importance/RNN_{VER}_{TF}m_all_categories_agg_no_volume_FIXED.csv', index=False)
print('saved')

In [ ]:
import glob, pandas as pd
nm_files = sorted(glob.glob(f'{BASE}/amia/feature/survey_name_map_24*'))
print(nm_files)
nm = pd.concat([pd.read_csv(f) for f in nm_files], ignore_index=True) 

for stub in ['surv_disability_errands_alone',
             'surv_overall_health_average_pain_7']:
    print('\n===', stub, '===')
    print(nm[nm.iloc[:,0].astype(str).str.startswith(stub)].to_string(index=False))

In [ ]:
def block_perm_seq(seq_groups, static_groups, n_repeats=5, seed=42, batch_size=2048):
    rng = np.random.default_rng(seed)
    n = len(te); rows = []

    idx_of = {c: i for i, c in enumerate(static_cols)}
    sm = static_test.copy()
    for name, cols in tqdm(list(static_groups.items()), desc='static blocks'):
        idx = [idx_of[c] for c in cols if c in idx_of]
        if not idx: continue
        block, drops = sm[:, idx].copy(), []
        for _ in range(n_repeats):
            perm = rng.permutation(n)
            sm[:, idx] = block[perm]
            drops.append(base - average_precision_score(y[te], predict_proba_seq(te, sm, nr, batch_size=batch_size)))
        sm[:, idx] = block
        rows.append({'block': name, 'n_cols': len(idx), 'tower': 'static',
                     'drop_mean': float(np.mean(drops)), 'drop_std': float(np.std(drops))})

    for name, chans in tqdm(list(seq_groups.items()), desc='sequence blocks'):
        mask = np.zeros(D_seq, dtype=bool); mask[chans] = True
        drops = []
        for _ in range(n_repeats):
            perm_rows = te[rng.permutation(n)]
            drops.append(base - average_precision_score(
                y[te], predict_proba_seq(te, static_test, nr, group_mask=mask,
                                         perm_row_ids=perm_rows, batch_size=batch_size)))
        rows.append({'block': name, 'n_cols': len(chans), 'tower': 'sequence',
                     'drop_mean': float(np.mean(drops)), 'drop_std': float(np.std(drops))})

    return pd.DataFrame(rows).sort_values('drop_mean', ascending=False).reset_index(drop=True)

In [ ]:
import json
seq_groups_fine = {}
for mod, (a, b) in modality_range.items():
    codes = json.load(open(f'{SEQ_DIR}/vocab_{mod}_{TF}.json'))['codes']
    assert len(codes) == (b - a)
    for local_idx, code in enumerate(codes):
        seq_groups_fine[f'{mod}__{code}'] = [a + local_idx]
print('sequence 组数:', len(seq_groups_fine))

In [ ]:
seq_only = block_perm_seq(seq_groups=seq_groups_fine, static_groups={}, n_repeats=5)
seq_only.to_csv(f'{BASE}/amia/perm_importance/RNN_{VER}_{TF}m_seq_concept.csv', index=False)  
print(seq_only.head(15).to_string(index=False))

In [ ]:
print(seq_only.shape)
print(seq_only.head())

In [ ]:
import os
os.makedirs(f'{BASE}/amia/perm_importance', exist_ok=True)
seq_only.to_csv(f'{BASE}/amia/perm_importance/RNN_{VER}_{TF}m_seq_concept.csv', index=False)
print('saved')

In [ ]:
os.makedirs(f'{BASE}/amia/perm_importance', exist_ok=True)
try: res_lgbm['sp']['top10'].to_csv(f'{BASE}/amia/perm_importance/LGB_{VER}_{TF}m_top10.csv', index=False)
except NameError: pass
try: res_lgbm['ehr']['top10'].to_csv(f'{BASE}/amia/perm_importance/LGB_ehrdemo_{TF}m_top10.csv', index=False)
except NameError: pass
try: res_rf['sp']['top10'].to_csv(f'{BASE}/amia/perm_importance/RF_{VER}_{TF}m_top10.csv', index=False)
except NameError: pass
try: full.to_csv(f'{BASE}/amia/perm_importance/RNN_{VER}_{TF}m_modality_level.csv', index=False)
except NameError: pass

print('checked')

In [ ]:
combined = pd.concat([top_rnn[top_rnn['tower'] == 'static'], seq_only], ignore_index=True)
combined = combined.sort_values('drop_mean', ascending=False).reset_index(drop=True)
combined['is_survey'] = combined['block'].str.startswith('surv_')

print(combined.head(15)[['block','n_cols','tower','drop_mean','drop_std','is_survey']].to_string(index=False))

import os
os.makedirs(f'{BASE}/amia/perm_importance', exist_ok=True)
combined.to_csv(f'{BASE}/amia/perm_importance/RNN_{VER}_{TF}m_combined_top.csv', index=False)
print('saved')

In [ ]:
print(combined.head(30)[['block','n_cols','tower','drop_mean','drop_std','is_survey']].to_string(index=False))